**初始化**

In [24]:
# 📌 第一區：初始化設定區（請先執行這一區）

# 套件安裝（首次執行）
!pip install yfinance gspread oauth2client ta schedule matplotlib
!pip install feedparser
# 匯入套件
import yfinance as yf
import pandas as pd
import numpy as np
from ta.momentum import RSIIndicator
from ta.trend import SMAIndicator
import datetime
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
import schedule
import time
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from bs4 import BeautifulSoup

# 🔑 Gmail 發信設定（請填入你剛剛產生的 App Password）
GMAIL_USER = 'ben12202004@gmail.com'
GMAIL_APP_PASSWORD = 'pichbddmefrhqaxk'

# 🧾 Google Sheets 設定
SPREADSHEET_NAME = 'Investment_Log'
TRANSACTION_SHEET = 'Transaction_History'
PORTFOLIO_SHEET = 'Portfolio_Status'

# 💰 投資組合設定（總投入金額會根據下方比例分配）
portfolio_allocation = {
    'VOO': 0.40,
    'SPMO': 0.10,
    '0050': 0.20,
    'VT': 0.20,
    'BND': 0.10
}

# 💸 自動抓取匯率（USDTWD）
def get_usd_to_twd_rate():
    try:
        fx = yf.download('USDTWD=X', period='1d', interval='1d')
        latest_rate = fx['Close'].iloc[-1]
        return round(float(latest_rate), 2)
    except Exception as e:
        print("⚠️ 匯率抓取失敗，使用預設值 32。錯誤：", e)
        return 30  # fallback 模擬值


# 📈 產生VOO走勢圖（近30日）
def generate_voo_chart(voo_df):
    plt.figure(figsize=(10, 6))
    plt.plot(voo_df['Date'], voo_df['Close'], label='VOO Close', linewidth=2)
    plt.plot(voo_df['Date'], voo_df['MA20'], label='MA20', linestyle='--')
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    plt.gcf().autofmt_xdate()
    plt.title('VOO Trend - Last 30 Days')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('voo_chart.png')
    plt.close()

# 自動更新的匯率與美金投入金額
usd_to_twd = get_usd_to_twd_rate()
monthly_investment_twd = 10000
monthly_investment_usd = round(monthly_investment_twd / usd_to_twd, 2)

# 🕐 排程設定（可用 Colab 設定 8:00 執行）
send_hour = 8

print("✅ 初始化完成，請繼續執行第二區資料採據")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 2.2 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=a572ee610d5d50a760e92d5289af7a4b884660315e5bc2ab5a1fefec4e3f1428
  Stored in directory: /root/.cache/pip/wheels/3b/25/2a/105d6a15df6914f4d15047691c6c28f9052cc1173e40285d03
Successfully built sgmllib3k


/tmp/ipython-input-24-844760806.py:47: FutureWarning: YF.download() has changed argument auto_adjust default to True
  fx = yf.download('USDTWD=X', period='1d', interval='1d')
[*********************100%***********************]  1 of 1 completed

✅ 初始化完成，請繼續執行第二區資料採據



/tmp/ipython-input-24-844760806.py:49: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return round(float(latest_rate), 2)


**抓取VOO市場資料與技術指標**


In [25]:
# 📊 第二區：抓取VOO市場資料與技術指標

def get_voo_data():
    today = datetime.datetime.today().date()
    start_date = today - datetime.timedelta(days=60)
    df = yf.download('VOO', start=start_date, end=today + datetime.timedelta(days=1))
    df = df[['Close']].copy()
    df['Date'] = df.index
    df['Close'] = df['Close'].astype(float)
    df['MA20'] = SMAIndicator(close=pd.Series(df['Close'].values.flatten()), window=20).sma_indicator().values
    df['RSI'] = RSIIndicator(close=pd.Series(df['Close'].values.flatten()), window=14).rsi().values
    df.reset_index(drop=True, inplace=True)
    return df

def get_previous_close():
    df = yf.download('VOO', period='5d', interval='1d')
    return df['Close'].iloc[-2]

def is_third_friday(date):
    if date.weekday() != 4:
        return False
    count = 0
    for day in range(1, 32):
        try:
            d = date.replace(day=day)
            if d.weekday() == 4:
                count += 1
            if d == date:
                return count == 3
        except:
            break
    return False

def get_month_high():
    today = datetime.date.today()
    first_day = today.replace(day=1)
    df = yf.download('VOO', start=first_day, end=today + datetime.timedelta(days=1))
    return df['Close'].max()

voo_df = get_voo_data()
latest_data = voo_df.iloc[-1]
close = latest_data['Close'].item() if hasattr(latest_data['Close'], 'item') else float(latest_data['Close'])
ma20 = latest_data['MA20'].item() if hasattr(latest_data['MA20'], 'item') else float(latest_data['MA20'])
rsi = latest_data['RSI'].item() if hasattr(latest_data['RSI'], 'item') else float(latest_data['RSI'])
today = datetime.date.today()

print(f"📈 最新VOO收盤價：{close:.2f}")
print(f"📊 RSI：{rsi:.2f} / MA20：{ma20:.2f}")


/tmp/ipython-input-25-225968477.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('VOO', start=start_date, end=today + datetime.timedelta(days=1))
[*********************100%***********************]  1 of 1 completed

📈 最新VOO收盤價：585.58
📊 RSI：77.52 / MA20：574.56


 📩 **第三區：根據條件寄出通知信件（條件、寄件、紀錄）**

In [26]:
# 📩 第三區：根據條件寄出通知信件（條件、寄件、紀錄）

def should_invest(close, ma20, rsi, today):
    prev_close = get_previous_close()
    pct_drop = float((prev_close - close) / prev_close)
    month_high = get_month_high()
    month_drop = float((month_high - close) / month_high)
    if today.day == 1 and pct_drop >= 0.01:
        return "[定額加碼提醒] 月初下跌超過1%"
    elif today.day != 1 and is_third_friday(today):
        return "[定額加碼提醒] 每月第三週五定投"
    elif month_drop >= 0.05:
        return "[跌幅加碼機會] 本月高點已下跌超過5%"
    elif rsi < 30 and close < ma20:
        return "[下跌加碼機會] VOO RSI < 30 且跌破 MA20"
    else:
        return None

def send_email(subject, body):
    msg = MIMEMultipart()
    msg['From'] = GMAIL_USER
    msg['To'] = GMAIL_USER
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'html'))

    try:
        with open("voo_chart.png", 'rb') as img_file:
            img = MIMEImage(img_file.read())
            img.add_header('Content-ID', '<voo_chart>')
            msg.attach(img)
        server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
        server.login(GMAIL_USER, GMAIL_APP_PASSWORD)
        server.send_message(msg)
        server.quit()
        print("📧 郵件已寄出")
    except Exception as e:
        print("❌ 郵件寄送失敗：", e)

# 🧾 將交易記錄寫入 Google Sheet
# ...（略，與原本相同）

# 產生通知內容
signal = should_invest(close, ma20, rsi, today)
macro_news = get_macro_news()
generate_voo_chart(voo_df)

if signal:
    subject = f"{signal} | 匯率: {usd_to_twd}"
    body = f"""
    <p>VOO 今日收盤價：{close:.2f}<br>
    RSI：{rsi:.2f} / MA20：{ma20:.2f}<br>
    建議：執行定額或加碼投資。<br>
    目前投入美金：{monthly_investment_usd} (約新台幣 {monthly_investment_twd})</p>
    <p>📉 VOO 近 30 日走勢圖：</p>
    <img src="cid:voo_chart" width="600">
    """
    send_email(subject, body)
    # ...（略）
else:
    subject = f"[每日投資提醒] VOO 無特殊買點 | 匯率: {usd_to_twd}"
    body = f"""
    <p>VOO 今日收盤價：{close:.2f}<br>
    RSI：{rsi:.2f} / MA20：{ma20:.2f}<br>
    建議：暫不加碼，維持定投。</p>
    <p>📉 VOO 近 30 日走勢圖：</p>
    <img src="cid:voo_chart" width="600">
    """
    send_email(subject, body)

/tmp/ipython-input-25-225968477.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('VOO', period='5d', interval='1d')
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-26-3981207097.py:5: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_drop = float((prev_close - close) / prev_close)
/tmp/ipython-input-25-225968477.py:37: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('VOO', start=first_day, end=today + datetime.timedelta(days=1))
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-26-3981207097.py:7: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  month_drop = float((month_high - close) / month_high)


📧 郵件已寄出


• NFL will fine more than 100 players for reselling Super Bowl tickets at a profit
• Auction sales fall 6% in the first half, raising fears of an art market shift
• FCC approves $8 billion Paramount-Skydance merger
